In [7]:
import torch
import torchvision
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
import torchvision.transforms as transforms

# Definicion de modulos
Hasta ahora hemos construidos redes definiendo explicitamente cada capa.
1. Comente la estructura de la siguiente red.

In [ ]:
class TemplateModule1(nn.Module):

    def __init__(self, in_channels: int, hidden_layer:int, n_classes: int, kernel_size: int = 3,
                 stride: int = 1, dropout: float = 0.1,padding=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, hidden_layer, kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm2d(hidden_layer)

        self.dropout = nn.Dropout2d(dropout)
        self.conv2 = nn.Conv2d(hidden_layer, hidden_layer, kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm2d(hidden_layer)
        self.bn3 = nn.BatchNorm1d(hidden_layer)
        self.fc1 = nn.Linear(hidden_layer, hidden_layer)
        self.fc2 = nn.Linear(hidden_layer, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dropout(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(F.relu(self.bn2(self.conv2(x))))
        x = F.avg_pool2d(x, x.size()[2:])
        x =  F.relu(self.bn3(self.fc1(x.view(len(x), -1))))
        x = self.fc2(x)
        return x

In [ ]:
def generate_loaders():
    # 1. Define transformations (convert to tensor and normalize)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # 2. Load the dataset
    train_dataset = torchvision.datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Create DataLoaders
    batch_size = 32
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, test_loader

def train(network: nn.Module, optimizer: nn.Module,  epoch:int,criterion:nn.Module, train_loader: DataLoader, train_losses:list):
    network.train() #Modo
    train_loss = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    network.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    for batchId, (data, target) in enumerate(train_loader): #Iterate over batches
        data = data.to(device)
        target = target.to(device)

        #Feedforward pass
        optimizer.zero_grad()
        output = network(data)
        # prob = output.softmax(dim=1)
        log_prob = output.log_softmax(dim=1)

        loss = criterion(log_prob, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    length = len(train_loader.dataset)
    train_loss /= length
    train_losses += [train_loss]


#Testing
def test(network: nn.Module, test_loader:DataLoader,criterion:nn.Module, test_losses:list, test_accuracy:list):
    network.eval() #Test mode
    test_loss = 0
    correct = 0
    mean_accuracy = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            target = target.to(device)
            output = network(data)
            # prob = output.softmax(dim=1)
            log_prob = output.log_softmax(dim=1)
            pred_class = torch.argmax(output,dim=1)
            mean_accuracy += torch.sum(pred_class==target)
            loss = criterion(log_prob, target)
            test_loss += loss.item()
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(target.data.view_as(pred)).sum()
    length = len(test_loader.dataset)
    test_loss /= length
    test_losses.append(test_loss)
    test_accuracy += [mean_accuracy.item()/length]


In [2]:
train_loader, test_loader = generate_loaders()

n_channels_input = 3
hidden_dim = 64
n_classes = 10
network = TemplateModule1(in_channels=n_channels_input, hidden_layer=hidden_dim,n_classes=n_classes)
optimizer = optim.SGD(network.parameters(),lr=0.01)
# optimizer = optim.Adam(network.parameters(),lr=0.001)
# Train the model
num_epochs = 10
log_interval = 10
accuracy_test = []

train_losses = []
train_counter = []
test_losses = []
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
network.to(device)
for epoch in tqdm(range(num_epochs)):
    train(network, optimizer, epoch,criterion, train_loader,train_losses)
    test(network,test_loader,criterion,test_losses,accuracy_test)
    train_counter += [epoch]
    if epoch % log_interval==0:
        print('Train epoch: {}  Loss train: {:.6f} loss test  {:.6f}'.format(epoch,  test_losses[-1], test_losses[-1]))
        torch.save(network.state_dict(),'model_{:d}.pth'.format(epoch))

In [ ]:
loss_train1= train_losses
loss_test1= test_losses
accuracy_test1= accuracy_test
epochs_1 = train_counter

# Modularizacion
La modularizacion es el acto de definir sub-modulos y usarlos para construir redes neuronales. Esto ayuda a facilitar la interpretacion y cohesion del modelo.
1. Analize el modulo TemplateModule2

In [ ]:
class TemplateModule2(nn.Module):

    def __init__(self, in_channels: int, hidden_layer: int, n_classes: int, kernel_size: int = 3,
                 stride: int = 1, dropout: float = 0.1, padding=1):
        super().__init__()
        self.block1 = ConvUnitModule(in_channels, hidden_layer,kernel_size= kernel_size,stride= stride,
                                     padding=padding,dropout=dropout)
        self.block2 = ConvUnitModule(hidden_layer, hidden_layer,kernel_size= kernel_size,stride= stride,
                                     padding=padding,dropout=dropout)
        self.bn_linear = nn.BatchNorm1d(hidden_layer)
        self.fc1 = nn.Linear(hidden_layer, hidden_layer)
        self.fc2 = nn.Linear(hidden_layer, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = F.avg_pool2d(x, x.size()[2:])
        x = F.relu(self.bn_linear(self.fc1(x.view(x.size(0), -1))))
        x = self.fc2(x)
        return x

class ConvUnitModule(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3,
                 stride: int = 1, padding: int | None = None, dropout: float = 0.1):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2  # "same" padding for odd kernels and stride 1
        # bias is redundant because BatchNorm adds its own shift
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(F.relu(self.bn(self.conv(x))))



In [ ]:
train_loader, test_loader = generate_loaders()

n_channels_input = 3
hidden_dim = 64
n_classes = 10
network = TemplateModule2(in_channels=n_channels_input, hidden_layer=hidden_dim,n_classes=n_classes)
optimizer = optim.SGD(network.parameters(),lr=0.01)
# optimizer = optim.Adam(network.parameters(),lr=0.001)
# Train the model
num_epochs = 10
log_interval = 10
accuracy_test = []

train_losses = []
train_counter = []
test_losses = []
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
network.to(device)
for epoch in tqdm(range(num_epochs)):
    train(network, optimizer, epoch,criterion, train_loader,train_losses)
    test(network,test_loader,criterion,test_losses,accuracy_test)
    train_counter += [epoch]
    if epoch % log_interval==0:
        print('Train epoch: {}  Loss train: {:.6f} loss test  {:.6f}'.format(epoch,  test_losses[-1], test_losses[-1]))
        torch.save(network.state_dict(),'model_{:d}.pth'.format(epoch))

In [ ]:
loss_train2 = train_losses
loss_test2 = test_losses
accuracy_test2 = accuracy_test
epochs_2 = train_counter

# Skip connections


In [6]:

class TemplateModule3(nn.Module):

    def __init__(self, in_channels: int, hidden_layer: int, n_classes: int, kernel_size: int = 3,
                 stride: int = 1, dropout: float = 0.1, padding=1):
        super().__init__()
        self.block1 = ConvSimpleSkip(in_channels, hidden_layer,kernel_size= kernel_size,stride= stride,
                                     padding=padding,dropout=dropout)
        self.bn_linear = nn.BatchNorm1d(hidden_layer)
        self.fc1 = nn.Linear(hidden_layer, hidden_layer)
        self.fc2 = nn.Linear(hidden_layer, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = F.avg_pool2d(x, x.size()[2:])
        x = F.relu(self.bn_linear(self.fc1(x.view(x.size(0), -1))))
        x = self.fc2(x)
        return x


class ConvSimpleSkip(nn.Module):


    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3,
                 stride: int = 1, padding=1, dropout: float = 0.1):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                               stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size,
                               stride=stride, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.dropout(F.relu(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        # the activation is applied after the sum, as in ResNet
        out = F.relu(out + self.shortcut(x))
        return self.dropout(out)

In [ ]:
train_loader, test_loader = generate_loaders()

n_channels_input = 3
hidden_dim = 64
n_classes = 10
network = TemplateModule3(in_channels=n_channels_input, hidden_layer=hidden_dim,n_classes=n_classes)
optimizer = optim.SGD(network.parameters(),lr=0.01)
# optimizer = optim.Adam(network.parameters(),lr=0.001)
# Train the model
num_epochs = 10
log_interval = 10
accuracy_test = []

train_losses = []
train_counter = []
test_losses = []
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
network.to(device)
for epoch in tqdm(range(num_epochs)):
    train(network, optimizer, epoch,criterion, train_loader,train_losses)
    test(network,test_loader,criterion,test_losses,accuracy_test)
    train_counter += [epoch]
    if epoch % log_interval==0:
        print('Train epoch: {}  Loss train: {:.6f} loss test  {:.6f}'.format(epoch,  test_losses[-1], test_losses[-1]))
        torch.save(network.state_dict(),'model_{:d}.pth'.format(epoch))

In [ ]:
loss_train2 = train_losses
loss_test2 = test_losses
accuracy_test2 = accuracy_test
epochs_2 = train_counter

1. Compare los tres modelos.

# Codigo
1. Escoja alguna red y realice una modificacion para generar dos ramas paralelas.
```
|        output          |
|          |             |
|        fuse            |
|       /     \          |
|    x1        x2        |
|     |        |         |
|  branch1  branch2      |
|      \      /          |
        Image

```

In [ ]:
class TemplateModule4(nn.Module):

    def __init__(self, in_channels: int, hidden_layer: int, n_classes: int, kernel_size: int = 3,
                 stride: int = 1, dropout: float = 0.1, padding=1):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:


# Transfer Learning
1. Que entiende por transfer learning?

In [ ]:
def transfer_weights(trained_network: nn.Module, recipient_network: nn.Module) -> None:
    """Copy parameters and buffers (e.g. BatchNorm running stats) from
    `trained_network` into `recipient_network` for every entry whose name and
    shape match. Entries that don't match are left untouched."""
    trained_state = trained_network.state_dict()
    recipient_state = recipient_network.state_dict()

    transferred, skipped = [], []
    for name, tensor in recipient_state.items():
        source = trained_state.get(name)
        if source is not None and source.shape == tensor.shape:
            recipient_state[name] = source.detach().clone().to(tensor.device, tensor.dtype)
            transferred.append(name)
        else:
            skipped.append(name)

    recipient_network.load_state_dict(recipient_state)
    print(f"Transferred {len(transferred)} tensors, skipped {len(skipped)}.")
    if skipped:
        print("Skipped:", ", ".join(skipped))

In [ ]:
class TemplateTransferLearning(nn.Module):

    def __init__(self, in_channels: int, hidden_layer: int,  kernel_size: int = 3,
                 stride: int = 1, dropout: float = 0.1, padding=1):
        super().__init__()
        self.block1 = ConvSimpleSkip(in_channels, hidden_layer,kernel_size= kernel_size,stride= stride,
                                     padding=padding,dropout=dropout)
        self.bn_linear = nn.BatchNorm1d(hidden_layer)
        self.fc1 = nn.Linear(hidden_layer, hidden_layer)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = F.avg_pool2d(x, x.size()[2:])
        x = F.relu(self.bn_linear(self.fc1(x.view(x.size(0), -1))))
        return x

def define_embeddings(hidden_state_network:nn.Module, dataset:torch.utils.data.Dataset, emb_size:int):
    """Run every element of `dataset` through `hidden_state_network` and return
    an (N, emb_size) matrix of embeddings and an (N,) array with the class of each element."""
    hidden_state_network.eval()  # BatchNorm uses running stats, dropout disabled
    device = next(hidden_state_network.parameters()).device
    # shuffle=False so that row i of the matrix corresponds to dataset[i]

    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    n = len(dataset)
    embeddings = np.zeros((n, emb_size), dtype=np.float32)
    classes = np.zeros(n, dtype=np.int64)

    start = 0
    with torch.no_grad():
        for data, target in tqdm(loader):
            data = data.to(device)
            output = hidden_state_network(data)
            end = start + len(data)
            embeddings[start:end] = output.view(len(data), -1).cpu().numpy()
            classes[start:end] = np.asarray(target)
            start = end
    return embeddings, classes


hidden_state_network = TemplateTransferLearning(3, 64)
transfer_weights(network,hidden_state_network)


In [ ]:

predict_embeddings_train,gt_classes_train = define_embeddings(hidden_state_network,train_loader.dataset,emb_size=hidden_dim)
predict_embeddings_test, gt_classes_test = define_embeddings(hidden_state_network, test_loader.dataset,
                                                               emb_size=hidden_dim)

model = LinearSVC()
model.fit(predict_embeddings_train, gt_classes_train)
predictions_accuracy =  accuracy_score(gt_classes_test, model.predict(predict_embeddings_test))
print(predictions_accuracy)

1. Como se compara el modelo con el anterior?